## Importing Packages

In [17]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms
import torchinfo
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
from torch.utils.data import Dataset

import helper_utils


In [18]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: CUDA")
else:
    device = torch.device("cpu")
    print(f"Using device: CPU")

Using device: CUDA


## Hyperparameters

In [19]:
BATCH_SIZE = 32
SEED = 42

## Datasets and DataLoadoaders

### Dataset Path

In [20]:
dataset_path = "data"

class_names = ['Agricultural', 'Baseball Diamond', 'Buildings', 'Dense Residential',
               'Harbor', 'Medium Residential', 'Mobile Home Park', 'Parking Lot',
               'Runway', 'Sparse Residential', 'Storage Tanks', 'Tennis Court', 
               'Airplane', 'Beach', 'Chaparral', 'Forest', 'Freeway', 'Golf Course',
               'Intersection', 'Overpass', 'River'
              ]

### Dataset Transformations

In [21]:
mean = [0.485, 0.490, 0.451]
std = [0.214, 0.197, 0.191]

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])


val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean,std=std),
])

### Dataset Wrapper

In [22]:
class DenseDataset(Dataset):
    def __init__(self, data, transform = None):

        self.data = data
        self.transform = transform
        self.classes = data.dataset.classes

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):

        img, label = self.data[index]

        if self.transform:
            img = self.transform(img)

            return img, label

### Creating datasets

In [23]:
UC_Merced_dataset = ImageFolder(root = dataset_path, transform = None)

train_size = int(0.7 * (len(UC_Merced_dataset)))
test_size = int(0.15 * (len(UC_Merced_dataset)))
val_size = len(UC_Merced_dataset) - train_size - test_size

train_data, test_data, val_data = random_split(UC_Merced_dataset, [train_size, test_size, val_size], generator=torch.Generator().manual_seed(SEED))

train_dataset = DenseDataset(train_data, transform = train_transform)
test_dataset = DenseDataset(test_data, transform = val_transform)
val_dataset = DenseDataset(val_data, transform = val_transform)

num_classes = len(train_dataset.classes)

# Print dataset summary
print(f"Number of classes:      {num_classes}")
print(f"Training set size:      {len(train_dataset)}")
print(f"Validation set size:    {len(val_dataset)}")
print(f"Test set size:          {len(test_dataset)}")

Number of classes:      21
Training set size:      1470
Validation set size:    315
Test set size:          315


# Dummy Classes

In [24]:
class Bottleneck(nn.Sequential):
    pass

class FeatureExtractor(nn.Sequential):
    pass


# Dense Layer Class

In [25]:
class DenseLayer(nn.Module):
    def __init__(self, in_channels, growth_rate, bn_size):
        super(DenseLayer, self).__init__()

        self.in_channels = in_channels
        self.growth_rate = growth_rate

        # Bottleneck layer
        # 1 x 1 convolution to reduce the number of channels before the 3 x 3 convolution
        self.bottleneck = Bottleneck(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace = True),
            nn.Conv2d(in_channels = in_channels, out_channels = bn_size * growth_rate, kernel_size = 1, stride = 1, bias = False)
        )

        # Feature Extractor layer
        # 3 x 3 convolution to extract features
        self.feature_extractor = FeatureExtractor(
            nn.BatchNorm2d(bn_size * growth_rate),
            nn.ReLU(inplace = True),
            nn.Conv2d(in_channels = bn_size * growth_rate, out_channels = growth_rate, stride = 1, kernel_size = 3, padding = 1, bias = False)
        )


    def forward(self, x):

        new_features = self.bottleneck(x)
        new_features = self.feature_extractor(new_features)

        x = torch.cat([x, new_features], dim = 1)

        return x

In [27]:
# Create an instance of the DenseLayer.
denselayer = DenseLayer(
    in_channels=3,      # Accepts an input with 3 channels (e.g., RGB).
    growth_rate=12,     # Will produce 12 new feature maps.
    bn_size=4           # The bottleneck layer will have (4 * 12) = 48 channels.
)

# Define the shape for a single image (Channels, Height, Width).
img_shape = (3, 64, 64)

# Define the full input shape for a batch of images.
input_size =  (BATCH_SIZE, *img_shape)

In [28]:
# Define a configuration dictionary to store parameters for the model summary.
config = {
    "input_size": input_size,
    "attr_names": ["input_size", "output_size", "num_params"],
    "col_names_display": ["Input Shape ", "Output Shape", "Param #"],
    "depth": 2
}

# Generate the model summary object using torchinfo with the specified configuration.
summary = torchinfo.summary(
    model=denselayer, 
    input_size=config["input_size"], 
    col_names=config["attr_names"], 
    depth=config["depth"]
)

# Display the summary as a styled HTML table.
print("--- Model Summary ---\n")
helper_utils.display_torch_summary(summary, config["attr_names"], config["col_names_display"], config["depth"])

--- Model Summary ---



Layer (type (var_name):depth-idx),Input Shape,Output Shape,Param #
DenseLayer (DenseLayer),"[32, 3, 64, 64]","[32, 15, 64, 64]",--
Bottleneck (bottleneck): 1-1,"[32, 3, 64, 64]","[32, 48, 64, 64]",--
BatchNorm2d (0): 2-1,"[32, 3, 64, 64]","[32, 3, 64, 64]",6
ReLU (1): 2-2,"[32, 3, 64, 64]","[32, 3, 64, 64]",--
Conv2d (2): 2-3,"[32, 3, 64, 64]","[32, 48, 64, 64]",144
FeatureExtractor (feature_extractor): 1-2,"[32, 48, 64, 64]","[32, 12, 64, 64]",--
BatchNorm2d (0): 2-4,"[32, 48, 64, 64]","[32, 48, 64, 64]",96
ReLU (1): 2-5,"[32, 48, 64, 64]","[32, 48, 64, 64]",--
Conv2d (2): 2-6,"[32, 48, 64, 64]","[32, 12, 64, 64]","5,184"
